In [1]:
# 전체 패키지 설치
# 최신 LangChain 및 관련 패키지 설치
# !pip install -q -U langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken
!pip install -q -U langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

문제) ChatPromptTemplate.from_messages()를 다루는 문제입니다. 아래의 조건에 맞는 내용을 작성하세요.

In [ ]:
# 심화 문제 3) (Markdown 출력)

# ============================================================
# TODO
#
# system
#
# "항상 아래 형식으로 답변한다.
#
# ## 개요
#
# ## 핵심 내용
#
# ## 예제
#
# ## 참고사항"
#
# human
#
# "{topic}"
#
# ============================================================

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        항상 아래 형식으로 답변한다.
        ## 개요
        ## 핵심 내용
        ## 예제
        ## 참고사항"
        """
    ),
    (
        "human", "{topic}"
    )
])


In [ ]:
# 심화 문제 4) (번역기)

# ============================================================
# TODO
#
# ChatPromptTemplate를 작성하시오.
#
# system
# "너는 전문 번역가이다."
#
# human
#
# 원문 :
# {text}
#
# 번역 언어 :
# {language}
#
# ============================================================

prompt = ChatPromptTemplate.from_messages([
    (
        "system", "너는 전문 번역가이다."
    ),
    (
        "human",
        """
        원문 :
        {text}

        번역 언어 :
        {language}

        """
    )
])

Few-shot Prompt
  - 언어 모델에 몇가지 예시를 제공하여 특정 작업을 수행하도록 유도하는 기법
  - 특정 도메인이나 형식의 질문에 대해 모델의 성능을 향상시키는데 효과적이다
  - 1. 왜 Few-shot Prompting을 사용하는가?
단순히 "이 질문에 답해줘"라고 하는 것(Zero-shot)보다, "이런 식으로 답해줘"라고 예시를 주는 것이 모델의 추론 일관성과 응답 품질을 비약적으로 높여줍니다.

  - 2. 최신 LangChain 방식의 구현
최신 LangChain에서는 FewShotChatMessagePromptTemplate을 사용하여 예시를 구조적으로 관리합니다.

In [4]:
from langchain_openai import ChatOpenAI
# 수정된 부분: langchain.prompts 대신 langchain_core.prompts 사용
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 예시 데이터 정의
examples = [
    {
        "question": "호날두로 삼행시 만들어 줘",
        "answer": "호: 호날두는 축구장에서 \n날 : 날로 먹고 튀었다. \n두 : 두번 다시 한국에 오지마."
    },
    {
        "question" : "김민재로 삼행시 만들어 줘",
        "answer": "김 : 김치는 맛있다. \n민 : 민족이 사랑하는 음식이다.\n재 : 재료는 배추이다."
    }
]



In [5]:
# 2. 개별 예시를 포맷팅할 템플릿 정의
exam_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="Question : {question}\n Answer : {answer}\n"
)


In [8]:
# 3. FewShotPromptTemplate 구성
# prefix를 추가하여 모델의 페르소나를 명확히 했습니다.
prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = exam_prompt,
    prefix = "당신은 재치 있는 삼행시 전문가입니다. 주어진 단어로 예시와 같은 형식의 삼행시를 작성하세요.",
    suffix = "Question : {input}\nAnswer : ",
    input_variables=["input"]
)



# 4. 모델 및 체인 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

chain = prompt | llm | StrOutputParser()


# 5. 실행
input_word = "손흥민으로 삼행시 만들어줘"

result = chain.invoke({"input" : input_word})


print("--- 완성된 프롬프트 ---")
print(prompt.format(input=input_word))
print("\n--- 모델의 답변 ---")
print(result)

--- 완성된 프롬프트 ---
당신은 재치 있는 삼행시 전문가입니다. 주어진 단어로 예시와 같은 형식의 삼행시를 작성하세요.

Question : 호날두로 삼행시 만들어 줘
 Answer : 호: 호날두는 축구장에서 
날 : 날로 먹고 튀었다. 
두 : 두번 다시 한국에 오지마.


Question : 김민재로 삼행시 만들어 줘
 Answer : 김 : 김치는 맛있다. 
민 : 민족이 사랑하는 음식이다.
재 : 재료는 배추이다.


Question : 손흥민
Answer : 

--- 모델의 답변 ---
손: 손흥민의 발끝에서  
흥: 흥미진진한 골이 터진다.  
민: 민첩한 드리블로 상대를 제친다.


In [10]:
# few-shot 템플릿을 사용 X
print(
llm.invoke("손흥민으로 삼행시 만들어줘").content
)

물론입니다! 손흥민으로 삼행시를 만들어볼게요.

손: 손끝으로 빚어낸  
흥: 흥미진진한 골장면,  
민: 민첩한 발놀림으로 승리로 이끈다!


In [11]:
from langchain_core.messages import HumanMessage

# 1. 프롬프트 포맷팅
prompt_text = prompt.format(input= "손흥민으로 삼행시 만들어줘")

# 2. invoke를 사용하여 모델 호출
# chatgpt([...]) 대신 chatgpt.invoke([...])를 사용하세요.
answer = llm.invoke([HumanMessage(content=prompt_text)])


# 3. 결과 출력
print(answer.content)

손: 손에 땀을 쥐게 하는  
흥: 흥미진진한 경기에서  
민: 민첩한 드리블로 골을 넣는다.


Few-shot 예제 포맷터 사용

In [15]:
# Few-shot 예제 포맷터 사용

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 모델 설정
chatgpt = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

In [16]:
# 2. 예제 세트 생성
examples = [
    {
        'question': '지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가?',
        'answer': '지구 대기의 약 78%를 차지하는 질소이다'
    },
    {
        "question": "광합성에 필요한 주요 요소들은 무엇인가요?",
        "answer": "광합성에 필요한 주요 요소는 빛, 이산화탄소, 물입니다."
    },
    {
        "question": "피타고라스 정리를 설명해주세요.",
        "answer": "피타고라스 정리는 직각삼각형에서 빗변의 제곱이 다른 두 변의 제곱의 합과 같다는 것입니다."
    },
    {
        "question": "지구의 자전 주기는 얼마인가요?",
        "answer": "지구의 자전 주기는 약 24시간(정확히는 23시간 56분 4초)입니다."
    },
    {
        "question": "DNA의 기본 구조를 간단히 설명해주세요.",
        "answer": "DNA는 두 개의 폴리뉴클레오티드 사슬이 이중 나선 구조를 이루고 있습니다."
    },
    {
        "question": "원주율(π)의 정의는 무엇인가요?",
        "answer": "원주율(π)은 원의 지름에 대한 원의 둘레의 비율입니다."
    }
]

In [17]:
# 3. 개별 예제 포맷터 (질문과 답변의 형태 정의)
example_prompt = PromptTemplate.from_template('질문 : {question}\n답변 : {answer}\n')

# 4. FewShotPromptTemplate 생성
# -> Few-Shot 프롬프트 템플릿 생성 후 새로운 질문에 대해 프롬프트 생성

prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,
    prefix = "당신은 친절한 과학 도우미입니다. 아래 예시와 같이 명확하고 간결하게 답변하세요.",
    suffix = "질문 : {input}\n답변 :",
    input_variables= ["input"]
)



# 5. LCEL 체인 구성 (Prompt -> LLM -> Parser)
chain = prompt | chatgpt | StrOutputParser()


# 6. 실행
user_query = '화성의 표면이 붉은 이유는 뭘까?'
response = chain.invoke({"input" : user_query})


print(f"질문: {user_query}")
print(f"답변: {response}")

질문: 화성의 표면이 붉은 이유는 뭘까?
답변: 화성의 표면이 붉은 이유는 철산화물(녹슨 철) 때문입니다.


In [18]:
# 예제 선택기 사용
# -> 의미적 유사성을 기반으로 가장 관련성이 높은 예제를 선택
# -> SemanticSimilarityExampleSelector

# Chroma 벡터스토어를 사용할 때 필요한 패키지
!pip install --upgrade chromadb langchain_chroma -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
# ChromaDB가 내부적으로 사용하는 opentelemetry 버전 충돌 이
# 현재 설치된 충돌 패키지들을 제거하고 호환 버전으로 재설치
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc
!pip install --upgrade "opentelemetry-api==1.24.0" "opentelemetry-sdk==1.24.0" "opentelemetry-exporter-otlp-proto-grpc==1.24.0"
!pip install --upgrade chromadb langchain-chroma


Found existing installation: opentelemetry-api 1.24.0
Uninstalling opentelemetry-api-1.24.0:
  Successfully uninstalled opentelemetry-api-1.24.0
Found existing installation: opentelemetry-sdk 1.24.0
Uninstalling opentelemetry-sdk-1.24.0:
  Successfully uninstalled opentelemetry-sdk-1.24.0
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.24.0
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.24.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.24.0
  Using cached opentelemetry_api-1.24.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached opentelemetry_sdk-1.24.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.24.0-py3-none-any.whl.metadata (2.2 kB)
Using cached opentelemetry_api-1.24.0-py3-none-any.whl (60 kB)
Using cached opentelemetry_sdk-1.24.0-py3-none-any.whl (106 kB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.24.0-py3-none-any.whl (18 kB)
ERROR: pip's dependency resolver does not cur

In [19]:
# Chroma
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser # StrOutputParser 임포트 추가

import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 1. 예제 데이터 (앞서 정의한 리스트 사용)
examples = [
    {'question': '지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가?', 'answer': '지구 대기의 약 78%를 차지하는 질소이다.'},
    {'question': '광합성에 필요한 주요 요소들은 무엇인가요?', 'answer': '광합성에 필요한 주요 요소는 빛, 이산화탄소, 물입니다.'},
    {'question': '피타고라스 정리를 설명해주세요.', 'answer': '피타고라스 정리는 직각삼각형에서 빗변의 제곱이 다른 두 변의 제곱의 합과 같다는 것입니다.'},
    {'question': '지구의 자전 주기는 얼마인가요?', 'answer': '지구의 자전 주기는 약 24시간(정확히는 23시간 56분 4초)입니다.'},
    {'question': 'DNA의 기본 구조를 간단히 설명해주세요.', 'answer': 'DNA는 두 개의 폴리뉴클레오티드 사슬이 이중 나선 구조를 이루고 있습니다.'},
    {'question': '원주율(π)의 정의는 무엇인가요?', 'answer': '원주율(π)은 원의 지름에 대한 원의 둘레의 비율입니다.'}
]

# 2. 예제 포맷터
example_prompt = PromptTemplate.from_template("질문 : {question}\n답변 : {answer}")


# 3. 예제 선택기 생성 (k값을 조정하여 예제 개수 설정 가능)
# SemanticSimilarityExampleSelector 초기화
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples = examples,
    embeddings= OpenAIEmbeddings(),
    vectorstore_cls= Chroma,
    k= 2,
)



# 4. FewShotPromptTemplate에 직접 selector 전달
prompt = FewShotPromptTemplate(
    example_selector= example_selector,
    example_prompt = example_prompt,
    prefix= "당신은 과학 전문가입니다. 아래에 질문에 대해 예시를 참고하여 답변하세요.",
    suffix= "질문 : {input}\n답변 : ",
    input_variables= ["input"]
)


# 5. 모델 및 체인 연결 (LCEL)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

chain = prompt | llm | StrOutputParser()

# 6. 실행
user_input = "화성의 표면이 붉은 이유는?"
response = chain.invoke({"input" : user_input})


print(f"--- 생성된 프롬프트 ---\n\n{prompt.invoke({"input" : user_input}).to_string()}\n")
print(f"--- 모델의 답변 ---\n{response}")


--- 생성된 프롬프트 ---

당신은 과학 전문가입니다. 아래에 질문에 대해 예시를 참고하여 답변하세요.

질문 : 지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가?
답변 : 지구 대기의 약 78%를 차지하는 질소이다.

질문 : 광합성에 필요한 주요 요소들은 무엇인가요?
답변 : 광합성에 필요한 주요 요소는 빛, 이산화탄소, 물입니다.

질문 : 화성의 표면이 붉은 이유는?
답변 : 

--- 모델의 답변 ---
화성의 표면이 붉은 이유는 주로 철 산화물, 즉 녹슨 철이 존재하기 때문이다. 이 철 산화물이 햇빛을 반사하여 붉은 색을 띠게 만든다.


Few-Shot 학습의 모델 성능을 향상시키는 방법
  - 고정예제 사용
  - 동적예제 사용

In [23]:
# 고정 예제 사용

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 예제 데이터
examples = [
  {"input" : "지구의 대기 중에 가장 많은 비율을 차지하는 기체는 무엇인가요?", "output":"질소입니다."},
  {"input" : "광합성에 필요한 중요 요소들은 무엇인가?", "output":"빛, 이산화탄소, 물입니다."}
]

# 2. 예제 템플릿: Chat 모델에 맞춰 메시지 형태로 정의
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])


# 3. Few-shot 프롬프트 템플릿
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples= examples,
)


# 4. 최종 프롬프트 조합
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 수학적 사고가 뛰어난 과학 전문가입니다."),
    few_shot_prompt,
    ("human", "{input}")
])


# 5. 모델 및 체인 구성 (LCEL)
# StrOutputParser()를 추가하여 결과값을 깔끔하게 문자열로만 받습니다.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

chain = prompt | model | StrOutputParser()


# 6. 실행
result = chain.invoke({'input': '지구의 자전 주기는 얼마인가?'})
print(result)

지구의 자전 주기는 약 24시간, 즉 1일입니다. 정확히는 약 23시간 56분 4초입니다.


최신 권장 코드 (LCEL + ChatPromptTemplate 통합)


In [28]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.output_parsers import StrOutputParser

# 1. 예제 데이터
examples = [
    {"input": "지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가요?", "output": "질소입니다."},
    {"input": "광합성에 필요한 주요 요소들은 무엇인가요?", "output": "빛, 이산화탄소, 물입니다."},
    {"input": "피타고라스 정리를 설명해주세요.", "output": "직각삼각형에서 빗변의 제곱은 다른 두 변의 제곱의 합과 같습니다."},
    {"input": "DNA의 기본 구조를 간단히 설명해주세요.", "output": "DNA는 이중 나선 구조를 가진 핵산입니다."},
    {"input": "원주율(π)의 정의는 무엇인가요?", "output": "원의 둘레와 지름의 비율입니다."},
]

# 2. 임베딩 모델 및 벡터 저장소 생성
embeddings = OpenAIEmbeddings()


# 3. 최신 방식의 예제 선택기 생성
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples = examples,
    embeddings = embeddings,
    vectorstore_cls = Chroma,
    k= 2,
    input_keys= ["input"],
)


# 4. Few-Shot 메시지 템플릿
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt = ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}")
    ]),
)



# 5. 최종 챗 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 수학적 사고가 뛰어난 과학 전문가 입니다."),
    few_shot_prompt,
    ("human", "{input}"),
])



# 6. 체인 구성 (LCEL)
model = ChatOpenAI(model= "gpt-4o-mini", temperature= 0.0)

chain = prompt | model | StrOutputParser()


# 7. 실행
result = chain.invoke({"input": "태양계에서 가장 큰 행성은 무엇인가?"})
print(result)

태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지름이 약 142,984킬로미터로, 태양계의 다른 행성들보다 훨씬 큽니다.


In [27]:
import langchain
import langchain_core
import langchain_community

print(langchain.__version__)
print(langchain_core.__version__)
print(langchain_community.__version__)

1.3.11
1.4.8
0.4.2


/tmp/ipykernel_1993/924729504.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


특징

- LangGraph + ChatOpenAI → ChatGPT 스타일 대화

- Chroma + SemanticSimilarityExampleSelector → 질문과 의미적으로 가까운 예제 자동 선택

- 멀티턴 대화 지원 → state["messages"]에 모든 대화 기록 유지

- 시각화는 주석 처리 → 설치 환경에 따라 문제 발생 가능하므로 생략

In [ ]:
!pip install -q langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 kB 9.8 MB/s eta 

In [29]:
!pip install -q langchain langchain-openai langgraph langchain_chroma chromadb openai

In [30]:
# ===== 설치 =====
# pip install langchain langchain-openai langgraph langchain_chroma chromadb openai

import os
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

# ===== OpenAI API Key =====


# ===== 상태 정의 =====
class State(TypedDict):
    messages : Annotated[List, add_messages]


# ===== LLM 준비 =====
llm = ChatOpenAI(model= "gpt-4o-mini", temperature= 0.7)


# ===== 예제 데이터 정의 =====
examples = [
    {"question" : "태양계에서 가장 큰 행성은?", "answer" : "목성입니다."},
    {"question" : "지구의 위성은?", "answer" : "달입니다."},
    {"question" : "화성이 붉은 이유는?", "answer" : "표면의 산화철 때문입니다."},
]




# ===== Chroma + SemanticSimilarityExampleSelector 초기화 =====
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples = examples,
    embeddings = embeddings,
    vectorstore_cls= Chroma,
    k= 1
)



# ===== 대화 처리 노드 정의 =====
def assistant_node(state: State):
    # 마지막 사용자 메시지 가져오기
    last_msg = state["messages"][-1].content if state["messages"] else ""


    # 유사 예제 선택
    selected_exam = example_selector.select_examples({"question" : last_msg})


    # 유사 예제 출력 (디버깅용)
    print("\n--------선택된 유사 예제---------")
    for examples in selected_exam:
      for k, v in examples.items():
        print(f"{k}: {v}")

    # LLM 호출
    response = llm.invoke(state["messages"])


    return {"messages" : [response]}

# ===== LangGraph 그래프 구성 =====
graph = StateGraph(State)
graph.add_node("assistant", assistant_node)
graph.add_edge(START, "assistant")
graph.add_edge("assistant", END)


# ===== 실행 가능한 그래프 빌드 =====
app = graph.compile()


# ===== ChatGPT 스타일 대화 실행 =====
inputs = {"messages": [HumanMessage(content="화성의 표면은?")]}

print("\n=== Assistant 응답 ===")
for output in app.stream(inputs):
    for state_name, state_data in output.items():
        for msg in state_data["messages"]:
            if hasattr(msg, "content"):
                print("Assistant:", msg.content)



=== Assistant 응답 ===

--------선택된 유사 예제---------
answer: 표면의 산화철 때문입니다.
question: 화성이 붉은 이유는?
Assistant: 화성의 표면은 다양한 지형과 특성을 가지고 있습니다. 주된 특징은 다음과 같습니다:

1. **산과 계곡**: 화성에는 태양계에서 가장 큰 화산인 올림푸스 몬스(Olympus Mons)와 깊은 계곡인 마리너스 트렌치(Valles Marineris)가 있습니다. 이 계곡은 지구의 그랜드 캐니언보다도 훨씬 큽니다.

2. **극관**: 화성의 극지방에는 얼음으로 이루어진 극관이 존재하며, 계절에 따라 크기가 변합니다. 여름에는 얼음이 녹아 일부 지역에서 물이 흐르는 증거가 발견되기도 했습니다.

3. **모래언덕과 평원**: 화성의 표면에는 넓은 평원과 모래언덕이 형성되어 있습니다. 이들은 바람에 의해 형성된 것으로, 화성의 대기와 기후가 영향을 미칩니다.

4. **운석의 흔적**: 화성의 표면에는 많은 운석 충돌의 흔적이 남아 있습니다. 이로 인해 화성의 표면은 다양한 크기의 크레이터로 뒤덮여 있습니다.

5. **색상과 조성**: 화성의 표면은 붉은색을 띠고 있는데, 이는 철 산화물(녹슨 철) 때문입니다. 이로 인해 "붉은 행성"이라는 별명을 가지고 있습니다.

화성의 표면은 과거에 물이 존재했을 가능성이 있는 증거를 가지고 있어, 우주 탐사와 생명체의 존재 여부에 대한 연구가 활발히 이루어지고 있습니다.


In [31]:
# ===== [추가] 그래프 시각화 및 저장 =====
try:
    # mermaid 형식의 이미지로 저장
    graph_image = app.get_graph().draw_mermaid_png()

    with open("assistant_graph.png", "wb") as f:
        f.write(graph_image)
    print("\n[알림] 그래프가 'assistant_graph.png'로 저장되었습니다.")
except Exception as e:
    print(f"\n[알림] 그래프 시각화에 실패했습니다: {e}")


[알림] 그래프가 'assistant_graph.png'로 저장되었습니다.


문제) Few-shot prompting (LCEL 최신 방식 기준)

In [33]:
# ============================================================
# FEW-SHOT LCEL MASTER EXERCISE
# 목표: example 기반 reasoning + LCEL 구조 이해
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o", temperature=0.2)

# ============================================================
# 0. FEW-SHOT EXAMPLES 정의
# ============================================================
# TODO 0:
# 아래 예시는 "문장 교정기"용 few-shot 데이터다.

examples = [
    # TODO: example 1
    # {"input": "...", "output": "..."}
    # {"input": "...", "output": "..."}
    {"input": "i goes to school yesterday", "output": "i went to school yesterday"},
    {"input": "she dont like apples", "output": "she doesn't like apples"},
    {"input": "he go home now", "output": "he is going home now"},
]

# ============================================================
# 1. FEW-SHOT PROMPT 구성
# ============================================================
# TODO 1:
# - example format 정의
# - system + few-shot + human 구조 구성

example_prompt = ChatPromptTemplate.from_messages([
    # TODO: input/output 구조 정의
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    # TODO: examples + example_prompt 연결
    examples= examples,
    example_prompt= example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    # TODO:
    # system + fewshot + human
    ("system", "너는 영어 문법 교정기다. 예시를 참고해서 문장들을 올바르게 교정을 해라."),
    few_shot_prompt,
    ("human", "{text}")
])

# ============================================================
# 2. LCEL CHAIN 구성
# ============================================================
# TODO 2:
# - prompt | model | parser 구조

chain = (
    # TODO
    final_prompt | model | StrOutputParser()
)

# ============================================================
# 3. 실행 테스트
# ============================================================
# TODO 3:

print(chain.invoke({"text": "i goes to school yesterday"}))
print(chain.invoke({"text": "she dont like apples"}))

I went to school yesterday.
she doesn't like apples
